
# Conversational Product Discovery MVP — Colab Prototype

This notebook is a **first vertical-slice MVP** for the Industry Project:

**Conversation → Structured Preference Update → Validated Preference State → Deterministic Candidate Retrieval → Representative Products → Grounded User Response**

The purpose is not to build the final AWS system yet. It is to verify that the core component works end-to-end before the code is split into modules and deployed.

## What this notebook tests

- Multi-turn preference accumulation
- Correction / overwrite
- Removal of a previous requirement
- Cross-turn retention
- Structured LLM output
- Deterministic state validation
- Deterministic catalogue filtering
- Simple representative-product selection
- Grounded final response generation
- Basic per-turn state accuracy against known ground truth

The notebook contains **three controlled demo conversations** with oracle states after every turn.

> Set `USE_OPENAI = True` to test the real LLM extractor.  
> Set `USE_OPENAI = False` to smoke-test the rest of the architecture without making API calls.


In [1]:
!pip -q install -U openai pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 124.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [2]:

import os
import json
import time
import copy
from getpass import getpass
from typing import Any, Dict, Optional

import pandas as pd

USE_OPENAI = True
MODEL = "gpt-5.6-terra"

if USE_OPENAI:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    from openai import OpenAI
    client = OpenAI()
else:
    client = None

print("USE_OPENAI =", USE_OPENAI)
print("MODEL =", MODEL)


Enter OPENAI_API_KEY: ··········
USE_OPENAI = True
MODEL = gpt-5.6-terra



## 1. Canonical preference state

For the first MVP, keep the schema intentionally small.

The LLM does **not** own the session state. It only returns a structured **update** for the current user turn. Deterministic application code validates and merges that update into the current state.


In [3]:

SUPPORTED_FIELDS = {
    "budget_max": "Maximum budget in AUD, numeric.",
    "brand": "Preferred laptop brand, e.g. Dell, Lenovo, ASUS.",
    "ram_min_gb": "Minimum RAM in GB.",
    "storage_min_gb": "Minimum storage in GB. Convert 1 TB to 1024 GB.",
    "screen_size_max_in": "Maximum screen size in inches.",
    "gpu_required": "Required GPU family/model, e.g. RTX 4060.",
    "usage": "Canonical usage: study, gaming, creative, business, or general.",
}

EMPTY_STATE = {
    "budget_max": None,
    "brand": None,
    "ram_min_gb": None,
    "storage_min_gb": None,
    "screen_size_max_in": None,
    "gpu_required": None,
    "usage": None,
}

def new_state():
    return copy.deepcopy(EMPTY_STATE)

def pretty_state(state):
    return json.dumps(state, indent=2, ensure_ascii=False)

print(pretty_state(new_state()))


{
  "budget_max": null,
  "brand": null,
  "ram_min_gb": null,
  "storage_min_gb": null,
  "screen_size_max_in": null,
  "gpu_required": null,
  "usage": null
}



## 2. Structured preference-update contract

Each LLM call returns only the changes expressed in the **latest** user message.

Example:

```json
{
  "updates": [
    {"field": "budget_max", "operation": "set", "value": "1500"},
    {"field": "ram_min_gb", "operation": "set", "value": "16"}
  ],
  "clarification_needed": false,
  "clarification_question": null
}
```

If the user says *"brand doesn't matter anymore"*, the expected update is:

```json
{"field": "brand", "operation": "remove", "value": null}
```


In [4]:

UPDATE_JSON_SCHEMA = {
    "type": "object",
    "properties": {
        "updates": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "field": {"type": "string", "enum": list(SUPPORTED_FIELDS.keys())},
                    "operation": {"type": "string", "enum": ["set", "remove"]},
                    "value": {"type": ["string", "null"]}
                },
                "required": ["field", "operation", "value"],
                "additionalProperties": False
            }
        },
        "clarification_needed": {"type": "boolean"},
        "clarification_question": {"type": ["string", "null"]}
    },
    "required": ["updates", "clarification_needed", "clarification_question"],
    "additionalProperties": False
}

EXTRACTION_INSTRUCTIONS = """
You are the natural-language preference-update extractor for a laptop product-discovery system.

Your job is ONLY to extract preference changes explicitly supported by the latest user message.

Rules:
1. Return only fields that the latest message changes. Do not repeat unchanged state.
2. Use operation="set" when the user introduces or changes a requirement.
3. Use operation="remove" only when the user explicitly revokes a previous requirement.
4. Never invent a requirement that was not stated.
5. Numeric values must be returned as plain strings containing the normalized number.
6. Convert 1 TB storage to 1024 GB, 2 TB to 2048 GB, etc.
7. Normalize usage to one of: study, gaming, creative, business, general.
8. For screen-size expressions such as "14 inches or smaller", update screen_size_max_in.
9. If a requirement cannot be represented by the supported schema, do not force it into another field.
10. Use clarification_needed only when the user's latest message is too ambiguous to safely update the state.
"""

def build_extraction_input(current_state, user_message):
    return f"""
SUPPORTED SCHEMA:
{json.dumps(SUPPORTED_FIELDS, indent=2)}

CURRENT VALIDATED STATE:
{json.dumps(current_state, indent=2)}

LATEST USER MESSAGE:
{user_message}
""".strip()


## 3. Deterministic validation and state update

**LLM proposes an update → application code validates it → only validated values enter the state.**

In [5]:

def _parse_value(field: str, value: Optional[str]):
    if value is None:
        raise ValueError(f"{field}: set operation requires a value")

    if field == "budget_max":
        x = float(value)
        if not (0 < x <= 100000):
            raise ValueError("budget_max out of range")
        return x
    if field == "ram_min_gb":
        x = int(float(value))
        if not (1 <= x <= 512):
            raise ValueError("ram_min_gb out of range")
        return x
    if field == "storage_min_gb":
        x = int(float(value))
        if not (1 <= x <= 32768):
            raise ValueError("storage_min_gb out of range")
        return x
    if field == "screen_size_max_in":
        x = float(value)
        if not (8 <= x <= 25):
            raise ValueError("screen_size_max_in out of range")
        return x
    if field == "usage":
        x = value.strip().lower()
        allowed = {"study", "gaming", "creative", "business", "general"}
        if x not in allowed:
            raise ValueError(f"usage must be one of {sorted(allowed)}")
        return x
    if field == "brand":
        return value.strip()
    if field == "gpu_required":
        return value.strip().upper()
    raise ValueError(f"Unsupported field: {field}")


def apply_update(current_state, update_payload):
    state = copy.deepcopy(current_state)
    audit = []

    for item in update_payload["updates"]:
        field = item["field"]
        operation = item["operation"]
        raw_value = item["value"]

        if field not in SUPPORTED_FIELDS:
            audit.append({"field": field, "status": "rejected", "reason": "unsupported field"})
            continue

        try:
            if operation == "remove":
                old = state[field]
                state[field] = None
                audit.append({"field": field, "operation": "remove", "old": old, "new": None, "status": "accepted"})
            elif operation == "set":
                parsed = _parse_value(field, raw_value)
                old = state[field]
                state[field] = parsed
                audit.append({"field": field, "operation": "set", "old": old, "new": parsed, "status": "accepted"})
            else:
                raise ValueError(f"Unsupported operation: {operation}")
        except Exception as e:
            audit.append({"field": field, "operation": operation, "value": raw_value, "status": "rejected", "reason": str(e)})

    return state, audit



## 4. LLM adapter

The OpenAI path uses the Responses API with a strict JSON schema.  
The mock path simply returns the oracle update for each predefined demo turn so the rest of the notebook can be tested without API usage.


In [6]:

def extract_update_openai(current_state, user_message):
    start = time.perf_counter()

    response = client.responses.create(
        model=MODEL,
        instructions=EXTRACTION_INSTRUCTIONS,
        input=build_extraction_input(current_state, user_message),
        text={
            "format": {
                "type": "json_schema",
                "name": "preference_update",
                "schema": UPDATE_JSON_SCHEMA,
                "strict": True
            }
        }
    )

    latency_s = time.perf_counter() - start
    payload = json.loads(response.output_text)

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", None) if usage else None
    output_tokens = getattr(usage, "output_tokens", None) if usage else None

    return payload, {
        "latency_s": latency_s,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens
    }


def extract_update(current_state, user_message, oracle_update=None):
    if USE_OPENAI:
        return extract_update_openai(current_state, user_message)

    if oracle_update is None:
        raise ValueError("Mock mode requires oracle_update for predefined demo turns.")

    return copy.deepcopy(oracle_update), {"latency_s": 0.0, "input_tokens": 0, "output_tokens": 0}


## 5. Small synthetic laptop catalogue

This exists only to verify the pipeline. Later replace it with the cleaned Amazon Reviews 2023 laptop catalogue.

In [7]:

catalogue = pd.DataFrame([
    {"product_id":"L01","title":"Dell Inspiron 14 Plus","brand":"Dell","price":1399,"ram_gb":32,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEL ARC","usage_tags":["study","business","general"]},
    {"product_id":"L02","title":"Lenovo IdeaPad Pro 14","brand":"Lenovo","price":1499,"ram_gb":32,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["study","business","general"]},
    {"product_id":"L03","title":"ASUS Zenbook 14","brand":"ASUS","price":1599,"ram_gb":16,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["study","business","general"]},
    {"product_id":"L04","title":"HP Pavilion 14","brand":"HP","price":1199,"ram_gb":16,"storage_gb":512,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["study","general"]},
    {"product_id":"L05","title":"Acer Swift Go 14","brand":"Acer","price":1299,"ram_gb":16,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["study","business","general"]},
    {"product_id":"L06","title":"Dell Latitude 14","brand":"Dell","price":1899,"ram_gb":32,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["business"]},
    {"product_id":"L07","title":"Lenovo ThinkPad T14","brand":"Lenovo","price":1999,"ram_gb":32,"storage_gb":1024,"screen_in":14.0,"gpu":"INTEGRATED","usage_tags":["business"]},
    {"product_id":"L08","title":"ASUS TUF Gaming A15","brand":"ASUS","price":1899,"ram_gb":16,"storage_gb":1024,"screen_in":15.6,"gpu":"RTX 4060","usage_tags":["gaming"]},
    {"product_id":"L09","title":"MSI Katana 15","brand":"MSI","price":2099,"ram_gb":16,"storage_gb":1024,"screen_in":15.6,"gpu":"RTX 4060","usage_tags":["gaming"]},
    {"product_id":"L10","title":"Lenovo Legion 5","brand":"Lenovo","price":2299,"ram_gb":32,"storage_gb":1024,"screen_in":16.0,"gpu":"RTX 4060","usage_tags":["gaming","creative"]},
    {"product_id":"L11","title":"ASUS ROG Zephyrus G14","brand":"ASUS","price":2399,"ram_gb":32,"storage_gb":1024,"screen_in":14.0,"gpu":"RTX 4060","usage_tags":["gaming","creative"]},
    {"product_id":"L12","title":"Acer Nitro V 15","brand":"Acer","price":1799,"ram_gb":16,"storage_gb":1024,"screen_in":15.6,"gpu":"RTX 4060","usage_tags":["gaming"]},
    {"product_id":"L13","title":"HP Envy 16 Creator","brand":"HP","price":2199,"ram_gb":32,"storage_gb":2048,"screen_in":16.0,"gpu":"RTX 4050","usage_tags":["creative","business"]},
    {"product_id":"L14","title":"Dell XPS 14","brand":"Dell","price":2499,"ram_gb":32,"storage_gb":1024,"screen_in":14.5,"gpu":"RTX 4050","usage_tags":["creative","business"]},
])

catalogue


,product_id,title,brand,price,ram_gb,storage_gb,screen_in,gpu,usage_tags
0,L01,Dell Inspiron 14 Plus,Dell,1399,32,1024,14.0,INTEL ARC,"[study, business, general]"
1,L02,Lenovo IdeaPad Pro 14,Lenovo,1499,32,1024,14.0,INTEGRATED,"[study, business, general]"
2,L03,ASUS Zenbook 14,ASUS,1599,16,1024,14.0,INTEGRATED,"[study, business, general]"
3,L04,HP Pavilion 14,HP,1199,16,512,14.0,INTEGRATED,"[study, general]"
4,L05,Acer Swift Go 14,Acer,1299,16,1024,14.0,INTEGRATED,"[study, business, general]"
5,L06,Dell Latitude 14,Dell,1899,32,1024,14.0,INTEGRATED,[business]
6,L07,Lenovo ThinkPad T14,Lenovo,1999,32,1024,14.0,INTEGRATED,[business]
7,L08,ASUS TUF Gaming A15,ASUS,1899,16,1024,15.6,RTX 4060,[gaming]
8,L09,MSI Katana 15,MSI,2099,16,1024,15.6,RTX 4060,[gaming]
9,L10,Lenovo Legion 5,Lenovo,2299,32,1024,16.0,RTX 4060,"[gaming, creative]"


## 6. Deterministic candidate retrieval

There is no learned ranker here. The validated state is translated directly into catalogue constraints.

In [8]:

def retrieve_candidates(state, df):
    out = df.copy()

    if state["budget_max"] is not None:
        out = out[out["price"] <= state["budget_max"]]
    if state["brand"] is not None:
        out = out[out["brand"].str.lower() == state["brand"].lower()]
    if state["ram_min_gb"] is not None:
        out = out[out["ram_gb"] >= state["ram_min_gb"]]
    if state["storage_min_gb"] is not None:
        out = out[out["storage_gb"] >= state["storage_min_gb"]]
    if state["screen_size_max_in"] is not None:
        out = out[out["screen_in"] <= state["screen_size_max_in"]]
    if state["gpu_required"] is not None:
        target = state["gpu_required"].upper()
        out = out[out["gpu"].str.upper().str.contains(target, regex=False)]
    if state["usage"] is not None:
        usage = state["usage"]
        out = out[out["usage_tags"].apply(lambda tags: usage in tags)]

    return out.sort_values(["price", "product_id"]).reset_index(drop=True)


def select_representatives(candidates, k=3):
    """Demonstration-only low/mid/high price selection. Not a recommendation ranker."""
    if len(candidates) <= k:
        return candidates.copy()
    positions = [0, len(candidates)//2, len(candidates)-1]
    positions = list(dict.fromkeys(positions))
    return candidates.iloc[positions].reset_index(drop=True)


## 7. Controlled demo conversations

Each oracle state is defined before the LLM is evaluated.

In [9]:

def set_update(field, value):
    return {"field": field, "operation": "set", "value": str(value)}

def remove_update(field):
    return {"field": field, "operation": "remove", "value": None}

def payload(*updates):
    return {"updates": list(updates), "clarification_needed": False, "clarification_question": None}

DEMOS = [
    {
        "name": "Demo 1 — accumulation + correction",
        "turns": [
            {"message": "I'm looking for a laptop for university.",
             "oracle_update": payload(set_update("usage", "study")),
             "expected_state": {**EMPTY_STATE, "usage":"study"}},
            {"message": "Keep it under $1,500 and I need at least 16 GB RAM.",
             "oracle_update": payload(set_update("budget_max", "1500"), set_update("ram_min_gb", "16")),
             "expected_state": {**EMPTY_STATE, "usage":"study", "budget_max":1500.0, "ram_min_gb":16}},
            {"message": "I'd like at least 512 GB of storage.",
             "oracle_update": payload(set_update("storage_min_gb", "512")),
             "expected_state": {**EMPTY_STATE, "usage":"study", "budget_max":1500.0, "ram_min_gb":16, "storage_min_gb":512}},
            {"message": "Actually, make the RAM 32 GB.",
             "oracle_update": payload(set_update("ram_min_gb", "32")),
             "expected_state": {**EMPTY_STATE, "usage":"study", "budget_max":1500.0, "ram_min_gb":32, "storage_min_gb":512}},
        ]
    },
    {
        "name": "Demo 2 — removal + cross-turn retention",
        "turns": [
            {"message": "I want a Dell laptop for business, under $2,000.",
             "oracle_update": payload(set_update("brand", "Dell"), set_update("usage", "business"), set_update("budget_max", "2000")),
             "expected_state": {**EMPTY_STATE, "brand":"Dell", "usage":"business", "budget_max":2000.0}},
            {"message": "It needs at least 1 TB of storage.",
             "oracle_update": payload(set_update("storage_min_gb", "1024")),
             "expected_state": {**EMPTY_STATE, "brand":"Dell", "usage":"business", "budget_max":2000.0, "storage_min_gb":1024}},
            {"message": "Actually, brand doesn't matter anymore.",
             "oracle_update": payload(remove_update("brand")),
             "expected_state": {**EMPTY_STATE, "brand":None, "usage":"business", "budget_max":2000.0, "storage_min_gb":1024}},
            {"message": "And I want something 14 inches or smaller.",
             "oracle_update": payload(set_update("screen_size_max_in", "14")),
             "expected_state": {**EMPTY_STATE, "brand":None, "usage":"business", "budget_max":2000.0, "storage_min_gb":1024, "screen_size_max_in":14.0}},
        ]
    },
    {
        "name": "Demo 3 — gaming constraints + budget overwrite",
        "turns": [
            {"message": "I need a gaming laptop with an RTX 4060.",
             "oracle_update": payload(set_update("usage", "gaming"), set_update("gpu_required", "RTX 4060")),
             "expected_state": {**EMPTY_STATE, "usage":"gaming", "gpu_required":"RTX 4060"}},
            {"message": "My maximum budget is $2,400.",
             "oracle_update": payload(set_update("budget_max", "2400")),
             "expected_state": {**EMPTY_STATE, "usage":"gaming", "gpu_required":"RTX 4060", "budget_max":2400.0}},
            {"message": "At least 16 GB RAM and 1 TB storage.",
             "oracle_update": payload(set_update("ram_min_gb", "16"), set_update("storage_min_gb", "1024")),
             "expected_state": {**EMPTY_STATE, "usage":"gaming", "gpu_required":"RTX 4060", "budget_max":2400.0, "ram_min_gb":16, "storage_min_gb":1024}},
            {"message": "That's too much — cap the budget at $2,000 instead.",
             "oracle_update": payload(set_update("budget_max", "2000")),
             "expected_state": {**EMPTY_STATE, "usage":"gaming", "gpu_required":"RTX 4060", "budget_max":2000.0, "ram_min_gb":16, "storage_min_gb":1024}},
        ]
    }
]

print("Demo conversations:", len(DEMOS))


Demo conversations: 3


## 8. Evaluation helpers

For the MVP: exact state match + active-field accuracy + latency/token usage.

In [10]:

def exact_state_match(pred, expected):
    return float(pred == expected)

def active_field_accuracy(pred, expected):
    active = [f for f in SUPPORTED_FIELDS if pred.get(f) is not None or expected.get(f) is not None]
    if not active:
        return 1.0
    return sum(pred.get(f) == expected.get(f) for f in active) / len(active)


## 9. Run the three conversations

In [12]:

all_turn_results = []
final_outputs = []

for demo_idx, demo in enumerate(DEMOS, start=1):
    print("=" * 100)
    print(demo["name"])
    print("=" * 100)
    state = new_state()

    for turn_idx, turn in enumerate(demo["turns"], start=1):
        predicted_update, meta = extract_update(
            current_state=state,
            user_message=turn["message"],
            oracle_update=turn["oracle_update"]
        )
        new_validated_state, audit = apply_update(state, predicted_update)
        expected_state = turn["expected_state"]

        exact = exact_state_match(new_validated_state, expected_state)
        active_acc = active_field_accuracy(new_validated_state, expected_state)
        candidates = retrieve_candidates(new_validated_state, catalogue)

        print(f"\nTurn {turn_idx}: {turn['message']}")
        print("Predicted update:")
        print(json.dumps(predicted_update, indent=2))
        print("Validated state:")
        print(pretty_state(new_validated_state))
        print(f"Candidate count: {len(candidates)}")
        print(f"Exact state match: {exact:.0%}")
        print(f"Active-field accuracy: {active_acc:.1%}")
        print(f"Latency: {meta['latency_s']:.2f}s")

        all_turn_results.append({
            "demo": demo_idx,
            "demo_name": demo["name"],
            "turn": turn_idx,
            "message": turn["message"],
            "exact_state_match": exact,
            "active_field_accuracy": active_acc,
            "candidate_count": len(candidates),
            "latency_s": meta["latency_s"],
            "input_tokens": meta["input_tokens"],
            "output_tokens": meta["output_tokens"],
        })
        state = new_validated_state

    final_candidates = retrieve_candidates(state, catalogue)
    representatives = select_representatives(final_candidates, k=3)
    final_outputs.append({"demo_name": demo["name"], "final_state": copy.deepcopy(state), "candidates": final_candidates, "representatives": representatives})

    print("\nFINAL QUALIFIED CANDIDATES")
    display(final_candidates[["product_id","title","price","ram_gb","storage_gb","screen_in","gpu"]])
    print("\nREPRESENTATIVE PRODUCTS")
    display(representatives[["product_id","title","price","ram_gb","storage_gb","screen_in","gpu"]])


Demo 1 — accumulation + correction

Turn 1: I'm looking for a laptop for university.
Predicted update:
{
  "updates": [
    {
      "field": "usage",
      "operation": "set",
      "value": "study"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated state:
{
  "budget_max": null,
  "brand": null,
  "ram_min_gb": null,
  "storage_min_gb": null,
  "screen_size_max_in": null,
  "gpu_required": null,
  "usage": "study"
}
Candidate count: 5
Exact state match: 100%
Active-field accuracy: 100.0%
Latency: 1.77s

Turn 2: Keep it under $1,500 and I need at least 16 GB RAM.
Predicted update:
{
  "updates": [
    {
      "field": "budget_max",
      "operation": "set",
      "value": "1500"
    },
    {
      "field": "ram_min_gb",
      "operation": "set",
      "value": "16"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated state:
{
  "budget_max": 1500.0,
  "brand": null,
  "ram_min_gb": 16,
  "storage_min_gb": null,
 

,product_id,title,price,ram_gb,storage_gb,screen_in,gpu
0,L01,Dell Inspiron 14 Plus,1399,32,1024,14.0,INTEL ARC
1,L02,Lenovo IdeaPad Pro 14,1499,32,1024,14.0,INTEGRATED



REPRESENTATIVE PRODUCTS


,product_id,title,price,ram_gb,storage_gb,screen_in,gpu
0,L01,Dell Inspiron 14 Plus,1399,32,1024,14.0,INTEL ARC
1,L02,Lenovo IdeaPad Pro 14,1499,32,1024,14.0,INTEGRATED


Demo 2 — removal + cross-turn retention

Turn 1: I want a Dell laptop for business, under $2,000.
Predicted update:
{
  "updates": [
    {
      "field": "brand",
      "operation": "set",
      "value": "Dell"
    },
    {
      "field": "usage",
      "operation": "set",
      "value": "business"
    },
    {
      "field": "budget_max",
      "operation": "set",
      "value": "2000"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated state:
{
  "budget_max": 2000.0,
  "brand": "Dell",
  "ram_min_gb": null,
  "storage_min_gb": null,
  "screen_size_max_in": null,
  "gpu_required": null,
  "usage": "business"
}
Candidate count: 2
Exact state match: 100%
Active-field accuracy: 100.0%
Latency: 1.43s

Turn 2: It needs at least 1 TB of storage.
Predicted update:
{
  "updates": [
    {
      "field": "storage_min_gb",
      "operation": "set",
      "value": "1024"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated 

...

,product_id,title,price,ram_gb,storage_gb,screen_in,gpu
0,L05,Acer Swift Go 14,1299,16,1024,14.0,INTEGRATED
1,L03,ASUS Zenbook 14,1599,16,1024,14.0,INTEGRATED
2,L07,Lenovo ThinkPad T14,1999,32,1024,14.0,INTEGRATED


Demo 3 — gaming constraints + budget overwrite

Turn 1: I need a gaming laptop with an RTX 4060.
Predicted update:
{
  "updates": [
    {
      "field": "usage",
      "operation": "set",
      "value": "gaming"
    },
    {
      "field": "gpu_required",
      "operation": "set",
      "value": "RTX 4060"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated state:
{
  "budget_max": null,
  "brand": null,
  "ram_min_gb": null,
  "storage_min_gb": null,
  "screen_size_max_in": null,
  "gpu_required": "RTX 4060",
  "usage": "gaming"
}
Candidate count: 5
Exact state match: 100%
Active-field accuracy: 100.0%
Latency: 2.62s

Turn 2: My maximum budget is $2,400.
Predicted update:
{
  "updates": [
    {
      "field": "budget_max",
      "operation": "set",
      "value": "2400"
    }
  ],
  "clarification_needed": false,
  "clarification_question": null
}
Validated state:
{
  "budget_max": 2400.0,
  "brand": null,
  "ram_min_gb": null,
  "storage_min_gb": 

,product_id,title,price,ram_gb,storage_gb,screen_in,gpu
0,L12,Acer Nitro V 15,1799,16,1024,15.6,RTX 4060
1,L08,ASUS TUF Gaming A15,1899,16,1024,15.6,RTX 4060



REPRESENTATIVE PRODUCTS


,product_id,title,price,ram_gb,storage_gb,screen_in,gpu
0,L12,Acer Nitro V 15,1799,16,1024,15.6,RTX 4060
1,L08,ASUS TUF Gaming A15,1899,16,1024,15.6,RTX 4060


## 10. Accuracy by conversation depth

Early check for whether state accuracy degrades as more turns accumulate.

In [13]:

results_df = pd.DataFrame(all_turn_results)
summary_by_turn = (
    results_df.groupby("turn", as_index=False)
    .agg(
        exact_state_accuracy=("exact_state_match", "mean"),
        active_field_accuracy=("active_field_accuracy", "mean"),
        mean_latency_s=("latency_s", "mean")
    )
)

display(results_df)
display(summary_by_turn)


,demo,demo_name,turn,message,exact_state_match,active_field_accuracy,candidate_count,latency_s,input_tokens,output_tokens
0,1,Demo 1 — accumulation + correction,1,I'm looking for a laptop for university.,1.0,1.0,5,1.767507,530,38
1,1,Demo 1 — accumulation + correction,2,"Keep it under $1,500 and I need at least 16 GB...",1.0,1.0,4,1.932220,540,55
2,1,Demo 1 — accumulation + correction,3,I'd like at least 512 GB of storage.,1.0,1.0,4,1.831201,538,41
3,1,Demo 1 — accumulation + correction,4,"Actually, make the RAM 32 GB.",1.0,1.0,2,2.068058,538,41
4,2,Demo 2 — removal + cross-turn retention,1,"I want a Dell laptop for business, under $2,000.",1.0,1.0,2,1.426766,536,64
5,2,Demo 2 — removal + cross-turn retention,2,It needs at least 1 TB of storage.,1.0,1.0,2,2.136248,538,42
6,2,Demo 2 — removal + cross-turn retention,3,"Actually, brand doesn't matter anymore.",1.0,1.0,6,4.528479,537,38
7,2,Demo 2 — removal + cross-turn retention,4,And I want something 14 inches or smaller.,1.0,1.0,6,1.038944,539,41
8,3,Demo 3 — gaming constraints + budget overwrite,1,I need a gaming laptop with an RTX 4060.,1.0,1.0,5,2.621928,534,54
9,3,Demo 3 — gaming constraints + budget overwrite,2,"My maximum budget is $2,400.",1.0,1.0,5,1.072462,536,40


,turn,exact_state_accuracy,active_field_accuracy,mean_latency_s
0,1,1.0,1.0,1.938734
1,2,1.0,1.0,1.713643
2,3,1.0,1.0,2.838328
3,4,1.0,1.0,1.690897



## 11. Grounded user-facing response

The response generator receives only the validated preference state and selected product records. It is explicitly instructed not to invent product facts.


In [14]:

RESPONSE_INSTRUCTIONS = """
You are the final product-discovery response generator.

Generate a concise user-facing response using ONLY the supplied validated preference state and product records.

Rules:
1. Do not invent product facts.
2. Every factual product claim must be directly supported by the supplied product records.
3. Do not claim battery life, build quality, performance benchmarks, discounts, availability, or other facts unless explicitly present.
4. Explain briefly how each shown product relates to the user's stated constraints.
5. If no products are supplied, state that no current catalogue products satisfy all active constraints and suggest that the user revise a constraint.
6. Do not claim that a product is objectively "best".
"""

def generate_grounded_response(state, representatives):
    if not USE_OPENAI:
        if representatives.empty:
            return "No current catalogue products satisfy all active constraints."
        lines = ["Qualified representative products:"]
        for _, r in representatives.iterrows():
            lines.append(f"- {r['title']}: AUD ${r['price']}, {r['ram_gb']} GB RAM, {r['storage_gb']} GB storage, {r['screen_in']}-inch display, GPU: {r['gpu']}.")
        return "\n".join(lines)

    records = representatives.to_dict(orient="records")
    response = client.responses.create(
        model=MODEL,
        instructions=RESPONSE_INSTRUCTIONS,
        input=f"""
VALIDATED PREFERENCE STATE:
{json.dumps(state, indent=2)}

SUPPLIED PRODUCT RECORDS:
{json.dumps(records, indent=2)}
""".strip()
    )
    return response.output_text


In [15]:

generated_responses = []

for output in final_outputs:
    response_text = generate_grounded_response(output["final_state"], output["representatives"])
    generated_responses.append({
        "demo_name": output["demo_name"],
        "state": output["final_state"],
        "products": output["representatives"].to_dict(orient="records"),
        "response": response_text,
    })

    print("=" * 100)
    print(output["demo_name"])
    print("-" * 100)
    print(response_text)
    print()


Demo 1 — accumulation + correction
----------------------------------------------------------------------------------------------------
Two options match your study-focused requirements: up to $1,500, at least 32 GB RAM, and at least 512 GB storage.

- **Dell Inspiron 14 Plus** — **$1,399**  
  32 GB RAM and 1 TB storage, with a 14-inch screen. It is tagged for study use.

- **Lenovo IdeaPad Pro 14** — **$1,499**  
  32 GB RAM and 1 TB storage, with a 14-inch screen. It is also tagged for study use.

Demo 2 — removal + cross-turn retention
----------------------------------------------------------------------------------------------------
Here are the current matches for your business laptop requirements: up to $2,000, at least 1TB storage, and a screen no larger than 14 inches.

- **Acer Swift Go 14** — **$1,299**  
  14-inch screen, 1TB storage, 16GB RAM, and tagged for business use.

- **ASUS Zenbook 14** — **$1,599**  
  14-inch screen, 1TB storage, 16GB RAM, and tagged for busines

## 12. Manual faithfulness inspection

For v1, inspect the generated response side-by-side with the exact records supplied to the response model.

In [16]:

for item in generated_responses:
    print("=" * 100)
    print(item["demo_name"])
    print("\nSOURCE PRODUCT RECORDS")
    print(json.dumps(item["products"], indent=2))
    print("\nGENERATED RESPONSE")
    print(item["response"])
    print()


Demo 1 — accumulation + correction

SOURCE PRODUCT RECORDS
[
  {
    "product_id": "L01",
    "title": "Dell Inspiron 14 Plus",
    "brand": "Dell",
    "price": 1399,
    "ram_gb": 32,
    "storage_gb": 1024,
    "screen_in": 14.0,
    "gpu": "INTEL ARC",
    "usage_tags": [
      "study",
      "business",
      "general"
    ]
  },
  {
    "product_id": "L02",
    "title": "Lenovo IdeaPad Pro 14",
    "brand": "Lenovo",
    "price": 1499,
    "ram_gb": 32,
    "storage_gb": 1024,
    "screen_in": 14.0,
    "gpu": "INTEGRATED",
    "usage_tags": [
      "study",
      "business",
      "general"
    ]
  }
]

GENERATED RESPONSE
Two options match your study-focused requirements: up to $1,500, at least 32 GB RAM, and at least 512 GB storage.

- **Dell Inspiron 14 Plus** — **$1,399**  
  32 GB RAM and 1 TB storage, with a 14-inch screen. It is tagged for study use.

- **Lenovo IdeaPad Pro 14** — **$1,499**  
  32 GB RAM and 1 TB storage, with a 14-inch screen. It is also tagged for study


## 13. MVP success criteria

The first prototype is successful if:

- The LLM returns valid structured updates.
- New requirements accumulate without deleting earlier requirements.
- Corrections overwrite the intended field only.
- Explicit removals clear the intended field only.
- Cross-turn requirements are retained.
- Invalid values are rejected by deterministic validation.
- Candidate retrieval changes mechanically with the validated state.
- The final response contains no unsupported product facts.

## Next step after this works

Do **not** move the notebook directly to AWS.

First refactor it into modules:

```text
src/
├── schemas/
├── llm/
├── state/
├── retrieval/
├── response/
├── evaluation/
└── api/
```

Then add FastAPI + unit tests + Docker locally. Only after that should the stable container/service be moved into the AWS MLOps workflow.
